<a href="https://colab.research.google.com/github/AndresDVS/Simulacion-de-Cajeros/blob/main/s25_m_dulo_2_actividad_did_ctica_2_m3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

class BankSimulation:
    def __init__(self, configuracion='1r2p', num_cajeros=3, horas_operacion=8):
        self.num_cajeros = num_cajeros
        self.tiempo_simulacion = horas_operacion * 60  # 480 minutos
        self.configuracion = configuracion # '1r2p' o '2r1p'

        # Asignación de roles a los cajeros según el escenario
        if configuracion == '1r2p':
            self.cajeros_roles = ['retiro', 'pago', 'pago']
        elif configuracion == '2r1p':
            self.cajeros_roles = ['retiro', 'retiro', 'pago']

        # Probabilidades globales
        self.prob_retiro = 0.7
        self.prob_pago = 0.3

        # Probabilidades por tipo de usuario (Tabla 2)
        self.prob_usuario_retiro = [0.23, 0.40, 0.17, 0.20]  # Rápido, Normal, Lento, Muy lento
        self.prob_usuario_pago = [0.10, 0.20, 0.30, 0.40]

        # Tiempos de servicio (Media en minutos - Tabla 1)
        self.servicio_retiro = [1, 2, 3, 4]
        self.servicio_pago = [3, 3, 5, 7]

        # Tiempos de llegada (Media en minutos - Tabla 1)
        self.llegada_retiro = [1, 2, 3, 3]
        self.llegada_pago = [1, 2, 3, 4]

    def generar_tiempo_exponencial(self, media):
        return np.random.exponential(media)

    def generar_tipo_accion(self):
        return 'retiro' if np.random.random() < self.prob_retiro else 'pago'

    def generar_tipo_usuario(self, tipo_accion):
        if tipo_accion == 'retiro':
            return np.random.choice([0, 1, 2, 3], p=self.prob_usuario_retiro)
        else:
            return np.random.choice([0, 1, 2, 3], p=self.prob_usuario_pago)

    def simular_dia(self, semilla=None):
        if semilla is not None:
            np.random.seed(semilla)

        clientes_atendidos = []
        cajeros_ocupados = [0] * self.num_cajeros

        # Colas independientes para cada tipo de acción
        cola_retiros = []
        cola_pagos = []

        eventos = []
        tiempo_actual = 0

        # Programar la primera llegada al banco
        tipo_accion = self.generar_tipo_accion()
        tipo_usuario = self.generar_tipo_usuario(tipo_accion)
        media_llegada = self.llegada_retiro[tipo_usuario] if tipo_accion == 'retiro' else self.llegada_pago[tipo_usuario]
        tiempo_llegada = self.generar_tiempo_exponencial(media_llegada)

        eventos.append((tiempo_llegada, 'llegada', tipo_accion, tipo_usuario, None))

        while tiempo_actual < self.tiempo_simulacion:
            if not eventos:
                break

            eventos.sort(key=lambda x: x[0])
            evento = eventos.pop(0)
            tiempo_actual = evento[0]

            if tiempo_actual > self.tiempo_simulacion:
                break

            tipo_evento = evento[1]

            if tipo_evento == 'llegada':
                tipo_accion = evento[2]
                tipo_usuario = evento[3]

                # Buscar un cajero disponible que atienda este tipo de acción
                cajero_disponible = None
                for i in range(self.num_cajeros):
                    if self.cajeros_roles[i] == tipo_accion and cajeros_ocupados[i] == 0:
                        cajero_disponible = i
                        break

                if cajero_disponible is not None:
                    cajeros_ocupados[cajero_disponible] = 1
                    media_serv = self.servicio_retiro[tipo_usuario] if tipo_accion == 'retiro' else self.servicio_pago[tipo_usuario]
                    tiempo_servicio = self.generar_tiempo_exponencial(media_serv)

                    eventos.append((tiempo_actual + tiempo_servicio, 'fin_servicio', tipo_accion, tipo_usuario, cajero_disponible))

                    clientes_atendidos.append({
                        'tipo_accion': tipo_accion,
                        'tipo_usuario': tipo_usuario,
                        'tiempo_espera': 0,
                        'tiempo_servicio': tiempo_servicio,
                        'cajero': cajero_disponible
                    })
                else:
                    # Si no hay cajero de su especialidad libre, se va a su cola respectiva
                    if tipo_accion == 'retiro':
                        cola_retiros.append((tiempo_actual, tipo_accion, tipo_usuario))
                    else:
                        cola_pagos.append((tiempo_actual, tipo_accion, tipo_usuario))

                # Programar la siguiente llegada general
                tipo_accion_nueva = self.generar_tipo_accion()
                tipo_usuario_nuevo = self.generar_tipo_usuario(tipo_accion_nueva)
                media_llegada_nueva = self.llegada_retiro[tipo_usuario_nuevo] if tipo_accion_nueva == 'retiro' else self.llegada_pago[tipo_usuario_nuevo]
                tiempo_llegada_sig = tiempo_actual + self.generar_tiempo_exponencial(media_llegada_nueva)

                eventos.append((tiempo_llegada_sig, 'llegada', tipo_accion_nueva, tipo_usuario_nuevo, None))

            elif tipo_evento == 'fin_servicio':
                cajero = evento[4]
                rol_cajero = self.cajeros_roles[cajero]

                # Seleccionar la cola que le corresponde a este cajero
                cola_actual = cola_retiros if rol_cajero == 'retiro' else cola_pagos

                if cola_actual:
                    cliente_cola = cola_actual.pop(0)
                    tiempo_llegada_cola = cliente_cola[0]
                    tipo_accion_cola = cliente_cola[1]
                    tipo_usuario_cola = cliente_cola[2]

                    tiempo_espera = tiempo_actual - tiempo_llegada_cola

                    media_serv = self.servicio_retiro[tipo_usuario_cola] if tipo_accion_cola == 'retiro' else self.servicio_pago[tipo_usuario_cola]
                    tiempo_servicio = self.generar_tiempo_exponencial(media_serv)

                    eventos.append((tiempo_actual + tiempo_servicio, 'fin_servicio', tipo_accion_cola, tipo_usuario_cola, cajero))

                    clientes_atendidos.append({
                        'tipo_accion': tipo_accion_cola,
                        'tipo_usuario': tipo_usuario_cola,
                        'tiempo_espera': tiempo_espera,
                        'tiempo_servicio': tiempo_servicio,
                        'cajero': cajero
                    })
                else:
                    cajeros_ocupados[cajero] = 0

        return pd.DataFrame(clientes_atendidos)

    def ejecutar_replicas(self, num_replicas=10):
        resultados = []
        for i in range(num_replicas):
            df = self.simular_dia(semilla=i)
            df['replica'] = i + 1
            resultados.append(df)
        return pd.concat(resultados, ignore_index=True)

# --- EJECUCIÓN DE AMBOS ESCENARIOS ---
if __name__ == "__main__":
    mapeo_usuarios = {0: 'Rápido', 1: 'Normal', 2: 'Lento', 3: 'Muy lento'}

    # 1. Simular Escenario A (1 Retiro + 2 Pagos)
    print("Simulando Escenario A...")
    sim_A = BankSimulation(configuracion='1r2p')
    res_A = sim_A.ejecutar_replicas(10)
    res_A['tipo_usuario'] = res_A['tipo_usuario'].map(mapeo_usuarios)
    res_A.to_excel('problemaBancario_Escenario_A.xlsx', index=False)

    # 2. Simular Escenario B (2 Retiros + 1 Pago)
    print("Simulando Escenario B...")
    sim_B = BankSimulation(configuracion='2r1p')
    res_B = sim_B.ejecutar_replicas(10)
    res_B['tipo_usuario'] = res_B['tipo_usuario'].map(mapeo_usuarios)
    res_B.to_excel('problemaBancario_Escenario_B.xlsx', index=False)

    print("\n¡Simulaciones completadas con éxito!")
    print("Archivos generados: 'problemaBancario_Escenario_A.xlsx' y 'problemaBancario_Escenario_B.xlsx'")

Simulando Escenario A...
Simulando Escenario B...

¡Simulaciones completadas con éxito!
Archivos generados: 'problemaBancario_Escenario_A.xlsx' y 'problemaBancario_Escenario_B.xlsx'


In [ ]:
import pandas as pd

# Cargar los resultados generados anteriormente
df_A = pd.read_excel('problemaBancario_Escenario_A.xlsx')
df_B = pd.read_excel('problemaBancario_Escenario_B.xlsx')

def analizar_escenario(df, nombre_escenario):
    print(f"=== ANÁLISIS DEL {nombre_escenario} ===")

    # 1. Cajero con menor y mayor tiempo promedio de atención
    print("\n1. Tiempo promedio de atención por cajero (Minutos):")
    atencion_cajero = df.groupby('cajero')['tiempo_servicio'].mean()
    print(atencion_cajero.round(2))

    # 2. Promedio de usuarios de cada tipo en la totalidad de cajeros (por réplica)
    print("\n2. Promedio de usuarios por tipo atendidos por día (en las 10 réplicas):")
    usuarios_por_replica = df.groupby(['replica', 'tipo_usuario']).size().unstack(fill_value=0)
    print(usuarios_por_replica.mean().round(2))

    # 3. Total de usuarios por réplica para ver el mínimo
    print("\n3. Total de usuarios totales por réplica (Para identificar el de menor cantidad):")
    total_por_replica = df.groupby('replica').size()
    print(total_por_replica)
    print(f"La réplica con MENOR cantidad de usuarios fue la #{total_por_replica.idxmin()} con {total_por_replica.min()} usuarios.")

    # 4. Tiempos promedio de espera global
    print("\n4. Tiempo promedio de espera global en cola (Minutos):")
    espera_global = df['tiempo_espera'].mean()
    print(f"{espera_global:.2f} minutos")
    print("-" * 40)
    return espera_global

# Ejecutar el análisis para ambos
espera_A = analizar_escenario(df_A, "ESCENARIO A (1 Caja Retiro / 2 Cajas Pago)")
espera_B = analizar_escenario(df_B, "ESCENARIO B (2 Cajas Retiro / 1 Caja Pago)")

print("\n CONCLUSIÓN AUTOMÁTICA PARA EL PUNTO 5:")
if espera_A < espera_B:
    print(f"El ESCENARIO A es el mejor porque los usuarios esperan menos en promedio ({espera_A:.2f} min vs {espera_B:.2f} min).")
else:
    print(f"El ESCENARIO B es el mejor porque los usuarios esperan menos en promedio ({espera_B:.2f} min vs {espera_A:.2f} min).")

=== ANÁLISIS DEL ESCENARIO A (1 Caja Retiro / 2 Cajas Pago) ===

1. Tiempo promedio de atención por cajero (Minutos):
cajero
0    2.32
1    5.29
2    5.10
Name: tiempo_servicio, dtype: float64

2. Promedio de usuarios por tipo atendidos por día (en las 10 réplicas):
tipo_usuario
Lento        45.8
Muy lento    51.2
Normal       68.2
Rápido       35.8
dtype: float64

3. Total de usuarios totales por réplica (Para identificar el de menor cantidad):
replica
1     192
2     202
3     204
4     221
5     182
6     230
7     204
8     196
9     186
10    193
dtype: int64
La réplica con MENOR cantidad de usuarios fue la #5 con 182 usuarios.

4. Tiempo promedio de espera global en cola (Minutos):
3.85 minutos
----------------------------------------
=== ANÁLISIS DEL ESCENARIO B (2 Cajas Retiro / 1 Caja Pago) ===

1. Tiempo promedio de atención por cajero (Minutos):
cajero
0    2.31
1    2.27
2    5.15
Name: tiempo_servicio, dtype: float64

2. Promedio de usuarios por tipo atendidos por día (en 